# Loading libraries

In [50]:
import pandas as pd
import numpy as np

# Loading datasets

In [51]:
## After filtering the 1000 Genomes populational dataset with bcftools to include only missense variants

pop_data_missense = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/populational_data_missense_only.tsv",
                       sep="\t"
                       )

/var/folders/v8/pzxy2t1n2_1901rt7g0cqp_00000gn/T/ipykernel_39363/3207912597.py:3: DtypeWarning: Columns (9,10,11,12,13,56,57,103,104,105,106,107,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  pop_data_missense = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/populational_data_missense_only.tsv",


# Data processing


In [ ]:
## Processing the hexadecimal representations
pop_data_missense.astype(str).stack().str.contains("x3b").any()


pop_data_missense = pop_data_missense.apply(
    lambda col: col.str.replace(r"\\x3b", "|", regex=True) if col.dtype == "object" else col
)

#### **Process the 8th column**




The dataset's 8th column comprises aggregated functional annotations. The primary objective of this processing phase is to extract and transform this aggregated information into multiple distinct features.

In [55]:
pop_data_missense.rename(columns={
    "'Functional annotations: 'Allele | Annotation | Annotation_Impact | Gene_Name | Gene_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO'":
    "vep_annotation"
    },
                         inplace=True
                         )

In [56]:
pop_data_missense_tmp = pop_data_missense.copy()
pop_data_missense_tmp["vep_annotation"] = pop_data_missense_tmp["vep_annotation"].str.split(",")

In [ ]:
def align_column(col, n):
    if pd.isna(col):
        return [pd.NA] * n

    if isinstance(col, str):
        if "/" in col:
            parts = col.split("/")
            return parts[:n] + [pd.NA] * max(0, n - len(parts))
        else:
            return [col] * n

    return [col] * n



rows = []

for _, row in pop_data_missense_tmp.iterrows():
    n = len(row["vep_annotation"])

    new_row = {}
    for col in pop_data_missense_tmp.columns:
        if col == "vep_annotation":
            new_row[col] = row[col]
        else:
            new_row[col] = align_column(row[col], n)

    rows.append(pd.DataFrame(new_row))


pop_data_missense_long = (
    pd.concat(rows)
      .explode("vep_annotation")
      .reset_index(drop=True)
)

In [ ]:
new_headers = [
    "Allele", "Annotation", "Annotation_Impact", "Gene_Name",
    "Gene_ID", "Feature_Type", "Feature_ID", "Transcript_BioType",
    "Rank", "HGVS_c", "HGVS_p", "cDNA_pos_len",
    "CDS_pos_len", "AA_pos_len", "Distance", "ERRORS_WARNINGS_INFO"
]

target_col = "vep_annotation"

split_df = pop_data_missense_long[target_col].str.split('|', expand=True)

split_df = split_df.iloc[:, :len(new_headers)]
split_df.columns = new_headers

col_index = pop_data_missense_long.columns.get_loc(target_col)

# The "slice and stitch" strategy
# Part A: columns from start up to (but not including) the target
part_a = pop_data_missense_long.iloc[:, :col_index]

# Part B: the new split columns (split_df)

# Part C: columns from immediately after the target to the end
part_c = pop_data_missense_long.iloc[:, col_index+1:]

# Combining them
pop_data_missense_long_split = pd.concat([part_a, split_df, part_c], axis=1)

pop_data_missense_long_split.head()

In [62]:
pop_data_missense_long_split.insert(
    0,
    "var_id",
    pop_data_missense_long_split[["CHROM","POS","REF","ALT"]].astype(str).agg(":".join, axis=1)
)


pop_data_missense_long_split["var_id"] = pop_data_missense_long_split["var_id"].str.replace("chr", "", regex=False)

In [63]:
pop_data_missense_long_split[["AA_pos", "AA_len"]] = pop_data_missense_long_split["AA_pos_len"].str.split("/", expand=True)
pop_data_missense_long_split = pop_data_missense_long_split.drop(columns="AA_len")

In [64]:
pop_data_missense_long_split = pop_data_missense_long_split.replace('.', np.nan)

In [ ]:
# Edit column "HGVS_p"
target_col = 'HGVS_p'

pattern = r'p\.([A-Z][a-z]+)\d+([A-Z][a-z]+)'
new_cols = pop_data_missense_long_split[target_col].str.extract(pattern)
new_cols.columns = ['Ref_AA', 'Alt_AA']  # Name the new columns

col_index = pop_data_missense_long_split.columns.get_loc(target_col)

# Part A: everything before HGVS_p
part_a = pop_data_missense_long_split.iloc[:, :col_index]

# Part B: the new columns (new_cols)

# Part C: everything after HGVS_p (using index+1 to skip the old column)
part_c = pop_data_missense_long_split.iloc[:, col_index+1:]

# 5. Concatenate them together
pop_data_missense_long_split = pd.concat([part_a, new_cols, part_c], axis=1)

# Result
pop_data_missense_long_split[['Ref_AA', 'Alt_AA']]

In [72]:
acmg_map = {'Benign': 0, 'Likely_benign': 1, 'Uncertain_significance': 2, 'Likely_pathogenic': 3, 'Pathogenic': 4}
polyphen_map = {'D': 1.0, 'P': 0.5, 'B': 0.0}


pop_data_missense_long_split['Polyphen2_HVAR_pred'] = pop_data_missense_long_split['Polyphen2_HVAR_pred'].map(polyphen_map)
pop_data_missense_long_split['Polyphen2_HDIV_pred'] = pop_data_missense_long_split['Polyphen2_HDIV_pred'].map(polyphen_map)
pop_data_missense_long_split['InterVar_automated'] = pop_data_missense_long_split['InterVar_automated'].map(acmg_map)

In [139]:
pop_data_missense_long_split = pop_data_missense_long_split[~pop_data_missense_long_split['Ref_AA'].isna()]

In [140]:
pop_data_missense_long_split.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/pop_data_missense_final.csv",
                                    sep="\t",
                                    index=False,
                                    na_rep="NA")